This notebook shows how to process associations to create a catalog

In [ ]:
import glob2
from datetime import timedelta, tzinfo,timezone
import multiprocessing as mp
from pathlib import Path
from collections import defaultdict
import os
from tqdm import tqdm
import pickle
import numpy as np
from matplotlib.patches import Ellipse
from matplotlib import pyplot as plt
import pandas as pd
from src.utils.physics.sound_model.quality_tester import LocalizationQualityTester, QualityLevel, QualityReport
from src.utils.physics.sound_model.spherical_sound_model import GridSphericalSoundModel as GridSoundModel
from src.utils.physics.sound_model.spherical_sound_model import HomogeneousSphericalSoundModel as HomogeneousSoundModel
from src.utils.physics.sound_model.ellipsoidal_sound_model import GridEllipsoidalSoundModel

In [ ]:
# paths
CATALOG_PATH = "../../../data/demo/data"
DETECTIONS_DIR = "../../../data/detection/TiSSNet/demo"
ASSOCIATION_OUTPUT_DIR = "../../../data/detection/association"
Path(f"{DETECTIONS_DIR}/cache").mkdir(parents=True, exist_ok=True)
Path(f"{ASSOCIATION_OUTPUT_DIR}/cache").mkdir(parents=True, exist_ok=True)
file_name = "demo"

with open(f"{DETECTIONS_DIR}/cache/STATIONS_{file_name}.npy", "rb") as f:
    STATIONS = pickle.load(f)

SOUND_MODEL = GridEllipsoidalSoundModel([f"../../../data/sound_model/min-velocities_month-{i:02d}.nc" for i in range(1,13)],loader ="netcdf")
# SOUND_MODEL = HomogeneousSoundModel(sound_speed=1485.5)

#Detections
with open(f"{DETECTIONS_DIR}/cache/IDX_TO_DET_{file_name}.npy", "rb") as f:
    IDX_TO_DET = pickle.load(f)
#grid
with open("../../../data/detection/association/grid/grid_to_coords_demo.pkl", "rb") as f:
    GRID_TO_COORDS = pickle.load(f)
lat_min, lon_min = GRID_TO_COORDS.min(axis=0)
lat_max, lon_max = GRID_TO_COORDS.max(axis=0)

#Associations from TAPAAs
files = glob2.glob(f"{DETECTIONS_DIR}/cache/associations_{file_name}.pkl")
with open(files[0], "rb") as f:
    associations = pickle.load(f)


Firstly, process all associations; at this point we apply the clock drift correction and time offset if any

In [ ]:
drift_ppm = {'ELAN': np.float64(-0.0222),
             'MADE': np.float64(0.),
             'MADW': np.float64(0.),
             'NEAMS': np.float64(0.),
             'RTJ': np.float64(0.),
             'SSEIR': np.float64(0.0),
             'SSWIR': np.float64(0.0346),
             'SWAMS-bot': np.float64(0.0048),
             'WKER2': np.float64(0.0)}

offset = {'ELAN': np.float64(0),
         'MADE' :  np.float64(0.0),
         'MADW' :  np.float64(0.0),
         'NEAMS':  np.float64(0.0),
         'RTJ'  :  np.float64(0.0),
         'SSEIR':  np.float64(0.0),
         'SSWIR':  np.float64(0.0),
         'SWAMS-bot': np.float64(0.0),
         'WKER2': np.float64(0.0)}


GLOBAL = (STATIONS, IDX_TO_DET, associations, GRID_TO_COORDS, lat_min, lon_min, lon_max, drift_ppm, offset)

def init_worker(global_data):
    global STATIONS, IDX_TO_DET, associations
    global GRID_TO_COORDS, lat_min, lon_min, lon_max
    global drift_ppm, offset
    (STATIONS, IDX_TO_DET, associations,
     GRID_TO_COORDS, lat_min, lon_min, lon_max,
     drift_ppm, offset) = global_data

def process(i):
    #Localise the ith association if more than 6 stations are present
    station = list(map(lambda j: STATIONS[j].get_pos(), associations[i][0][:,0]))
    if len(station) < 6:
        return None

    #List of detection time corrected from drift of each station
    det = list(map(lambda j: IDX_TO_DET[j][0], associations[i][0][:,1]))
    drift_errors = list(map(lambda j,k :timedelta(seconds=offset[STATIONS[j].name]) + timedelta(seconds=STATIONS[j].get_clock_error(k, drift_ppm=drift_ppm[STATIONS[j].name])),associations[i][0][:,0], det ))
    det = list(map(lambda d, e : d-e,det,drift_errors))

    #For weighted least square we need an uncertainty for each detection
    drift = list(map(
        lambda s, d: 0.1 + STATIONS[s].get_clock_error(IDX_TO_DET[d][0],drift_ppm=0.14) if "not_ok"  in STATIONS[s].other_kwargs.values()
                     else 0.1,
        associations[i][0][:,0],
        associations[i][0][:,1]
    ))

    detections_uncertanty = [3]*len(det) #taking pick uncertanty and T-wave path uncertainty as 2s

    #Initialisation
    c0 = list(map(lambda j: GRID_TO_COORDS[j], associations[i][1]))
    min_date = np.argmin(det)
    t0 = -1 * SOUND_MODEL.get_sound_travel_time(np.mean(c0, axis =0), station[min_date], det[min_date])
    x0 = [t0]+list(np.mean(c0, axis =0))

    #least squares using
    # /!\ only avaiable with GridEllipsoidalSoundModel
    res= SOUND_MODEL.localize_with_uncertainties(
        station, det,y_min=lon_min-6, x_min=lat_min-6,y_max=lon_max+6,x_max=lat_max+6, drift_uncertainties=drift,pick_uncertainties=detections_uncertanty, initial_pos=x0
    )
    # For other models uncomment the following line
    # res = SOUND_MODEL.localize_common_source( station, det,y_min=lon_min-6, x_min=lat_min-6,y_max=lon_max+6,x_max=lat_max+6,initial_pos=x0)

    return i, res


CHUNK_SIZE = 100
results = {}
with mp.Pool(max(1,mp.cpu_count()-5),initializer=init_worker,initargs=(GLOBAL,)) as pool :
    for r in tqdm(pool.imap(process, range(len(associations)), chunksize=CHUNK_SIZE),
                  total=len(associations)):
        if r is not None:
            i, res = r
            results[i] = res


with open(f"{DETECTIONS_DIR}/cache/results_{file_name}.pkl", "wb") as f:
    pickle.dump(results, f)

#Relaod previously calculated results
# with open(f'{DETECTIONS_DIR}/cache/results_{file_name}.pkl', 'rb') as f:
#     results = pickle.load(f)

Filter the associations to discar the ones that have common detections.

In [ ]:
def filter_by_cost_threshold(results_dict, cost_threshold=10000):
    """
    Filtre les associations par seuil de coût.

    Args:
        results_dict: Dictionnaire des résultats d'optimisation
        cost_threshold: Seuil de coût maximum

    Returns:
        dict: results_dict filtré par coût
    """
    print(f"Step 1: Filtering by cost threshold ({cost_threshold})...")

    filtered_results = {}
    for i, res in results_dict.items():
        cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)
        if cost < cost_threshold:
            filtered_results[i] = res

    print(f"Results after cost filtering: {len(filtered_results)}")
    return filtered_results

def remove_overlap_bipartite(results_dict, association,
                                       n_params=2,
                                       alpha_global=0.05,
                                       alpha_outlier=0.01,
                                       min_quality=QualityLevel.GOOD,
                                       bilateral_test=False,
                                       stations=None,
                                       gap_threshold=None):
    """
    Filtrage des localisations avec tests statistiques

    Args:
        results_dict: Dict {event_id: OptimizeResult}
        association: Dict {event_id: (array_associations, ...)}
        n_params: Nombre de paramètres (2 pour x,y)
        alpha_global: Niveau du test global (défaut 5%)
        alpha_outlier: Niveau pour outliers individuels (défaut 0.1%)
        min_quality: Niveau minimum acceptable (défaut MARGINAL)
        bilateral_test: False = unilatéral (rejette seulement σ² >> 1)
        stations: Dict/list des stations pour calcul azimuthal gap (optionnel)
        gap_threshold: Seuil max d'azimuthal gap en degrés (défaut 270°) #This is not use at all yet
    """
    print("=== Filtrage avec tests statistiques rigoureux ===")
    print(f"    Test global: {'bilatéral' if bilateral_test else 'unilatéral (σ² > 1 seulement)'}")
    if stations is not None:
        print(f"    Azimuthal gap: activé (seuil = {gap_threshold}°)")

    # Initialiser le testeur
    tester = LocalizationQualityTester(
        n_params=n_params,
        alpha_global=alpha_global,
        alpha_outlier=alpha_outlier,
        bilateral_test=bilateral_test
    )

    # === PHASE 1 : Évaluer toutes les localisations ===
    print("\n[1/4] Évaluation de la qualité des localisations...")
    events = {}
    det_to_events = defaultdict(set)

    quality_stats = {level: 0 for level in QualityLevel}

    for event_id in tqdm(results_dict.keys(), desc="Quality evaluation"):
        result = results_dict[event_id]

        # Extraire les détections associées
        try:
            arr = np.asarray(association[event_id][0])
            detections = arr[:, 1].astype(int) if arr.ndim == 2 and arr.size > 0 else []
        except Exception:
            detections = []

        det_set = frozenset(detections)

        # Évaluation rigoureuse
        quality_report = tester.evaluate(result)
        quality_stats[quality_report.level] += 1

        events[event_id] = {
            'detection_set': det_set,
            'quality': quality_report,
            'cost':result.cost,
            'n_stations': arr.shape[0] if arr.ndim == 2 else 0
        }

        for det in det_set:
            det_to_events[det].add(event_id)

    # Afficher statistiques de qualité
    print("\nDistribution des niveaux de qualité:")
    for level in QualityLevel:
        count = quality_stats[level]
        pct = 100 * count / len(events) if events else 0
        print(f"  {level.name:12s}: {count:5d} ({pct:5.1f}%)")

    # === PHASE 2 : Pré-filtrage par qualité minimale ===
    print(f"\n[2/4] Pré-filtrage (qualité >= {min_quality.name})...")

    valid_events = {
        eid for eid, ev in events.items()
        if ev['quality'].level.value >= min_quality.value
    }

    rejected_quality = len(events) - len(valid_events)
    print(f"  Rejetés pour qualité insuffisante: {rejected_quality}")

    # === PHASE 3 : Trouver composantes connexes ===
    print("\n[3/4] Identification des conflits...")

    # Union-Find
    parent = {i: i for i in valid_events}

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        parent[find(x)] = find(y)

    for det, event_set in det_to_events.items():
        valid_in_set = [e for e in event_set if e in valid_events]
        if len(valid_in_set) > 1:
            for i in range(1, len(valid_in_set)):
                union(valid_in_set[0], valid_in_set[i])

    # Grouper
    clusters_dict = defaultdict(set)
    for event_id in valid_events:
        clusters_dict[find(event_id)].add(event_id)

    clusters = list(clusters_dict.values())
    isolated = [list(c)[0] for c in clusters if len(c) == 1]
    conflicting = [c for c in clusters if len(c) > 1]

    print(f"  Événements isolés: {len(isolated)}")
    print(f"  Clusters conflictuels: {len(conflicting)}")

    # === PHASE 4 : Résolution des conflits ===
    print("\n[4/4] Résolution des conflits par sélection optimale...")

    def selection_key(event_id):
        """
        Clé de tri pour sélection (plus grand = meilleur)

        Priorité :
        1. Niveau de qualité (EXCELLENT > GOOD > ...)
        2. Test global passé
        3. Nombre d'observations (plus = mieux, plus robuste)
        4. σ² proche de 1 mais sans pénaliser σ² < 1 excessivement
        """
        ev = events[event_id]
        q = ev['quality']
        # Pour σ², pénaliser seulement > 1 (outliers/sous-estimation erreurs)
        # σ² < 1 est acceptable (surestimation conservatrice des erreurs)
        sigma2 = q.sigma2_hat if np.isfinite(q.sigma2_hat) else 100.0
        sigma2_penalty = max(0, sigma2 - 1.0)  # 0 si σ² <= 1, positif sinon

        return (
            q.level.value,                    # Niveau de qualité
            1 if q.global_test_passed else 0, # Test global
            # q.n_obs,                          # Nombre d'observations
            -len(q.outlier_indices),          # Moins d'outliers = mieux
            -sigma2_penalty,                  # Pénalité σ² > 1 seulement
            ev['n_stations']                  # Tie-breaker
        )

    #you can try diffenrent quality score
    # def selection_key(event_id):
    #     q = events[event_id]['quality']
    #     score = q.selection_score if np.isfinite(q.selection_score) else -np.inf
    #     return (q.level.value, score)
    # def selection_key(event_id):
    #     q = events[event_id]['cost']
    #     # score = q.selection_score if np.isfinite(q.selection_score) else -np.inf
    #     return (q)

    selected_events = set()
    selected_detections = set()
    selection_reasons = defaultdict(list)

    # Traiter les clusters conflictuels
    for cluster in tqdm(conflicting, desc="Resolving conflicts"):
        # Trier par qualité décroissante
        sorted_events = sorted(cluster, key=selection_key, reverse=True)

        cluster_selected = []
        cluster_used_dets = set()

        for event_id in sorted_events:
            event_dets = events[event_id]['detection_set']

            # Sélectionner si pas de conflit avec déjà sélectionnés
            if event_dets.isdisjoint(cluster_used_dets):
                cluster_selected.append(event_id)
                cluster_used_dets.update(event_dets)
                selection_reasons['selected_in_cluster'].append(event_id)
            else:
                # Conflit - vérifier si remplacement justifié
                overlap = event_dets & cluster_used_dets
                selection_reasons['rejected_overlap'].append(
                    (event_id, len(overlap))
                )

        selected_events.update(cluster_selected)
        selected_detections.update(cluster_used_dets)

    # Ajouter les événements isolés
    for event_id in sorted(isolated, key=selection_key, reverse=True):
        event_dets = events[event_id]['detection_set']
        if event_dets.isdisjoint(selected_detections):
            selected_events.add(event_id)
            selected_detections.update(event_dets)
            selection_reasons['isolated'].append(event_id)

    # === RÉSUMÉ ===
    print("\n" + "="*50)
    print("RÉSUMÉ DE LA SÉLECTION")
    print("="*50)
    print(f"Événements en entrée:     {len(results_dict)}")
    print(f"Rejetés (qualité):        {rejected_quality}")
    print(f"Événements sélectionnés:  {len(selected_events)}")
    print(f"Détections utilisées:     {len(selected_detections)}")



    # Statistiques sur les sélectionnés
    if selected_events:
        selected_qualities = [events[e]['quality'] for e in selected_events]

        sigma2_values = [q.sigma2_hat for q in selected_qualities if np.isfinite(q.sigma2_hat)]
        scores = [q.selection_score for q in selected_qualities if np.isfinite(q.selection_score)]

        print(f"\nQualité des sélectionnés:")
        print(f"  σ² moyen: {np.mean(sigma2_values):.3f} ± {np.std(sigma2_values):.3f}")
        print(f"  Score moyen: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

        level_counts = defaultdict(int)
        for q in selected_qualities:
            level_counts[q.level] += 1

        print(f"\n  Distribution:")
        for level in QualityLevel:
            if level_counts[level] > 0:
                print(f"    {level.name}: {level_counts[level]}")
    # Conserver la structure de sortie originale : {event_id: OptimizeResult}
    return {eid: results_dict[eid] for eid in selected_events if eid in results_dict}

def get_quality_diagnostics(results_dict, association, n_params=2,
                            alpha_global=0.05, alpha_outlier=0.001):
    """
    Génère un diagnostic de qualité pour tous les événements
    (fonction utilitaire séparée pour l'analyse)

    Args:
        results_dict: Dict {event_id: OptimizeResult}
        association: Dict {event_id: (array_associations, ...)}

    Returns:
        DataFrame avec diagnostics détaillés
    """
    tester = LocalizationQualityTester(n_params, alpha_global, alpha_outlier)

    rows = []
    for eid, result in results_dict.items():
        q = tester.evaluate(result)

        # Nombre de détections
        try:
            arr = np.asarray(association[eid][0])
            n_det = arr.shape[0] if arr.ndim == 2 else 0
        except:
            n_det = 0

        rows.append({
            'origine_time': result.x[0],
            'event_id': eid,
            'quality_level': q.level.name,
            'sigma2_hat': q.sigma2_hat,
            'global_test_passed': q.global_test_passed,
            'n_outliers': len(q.outlier_indices),
            'outlier_indices': q.outlier_indices,
            'n_obs': q.n_obs,
            'dof': q.dof,
            'selection_score': q.selection_score,
            'chi2_lower': q.chi2_bounds[0],
            'chi2_upper': q.chi2_bounds[1],
            'n_detections': n_det
        })

    return pd.DataFrame(rows)




def filter_associations_simple(results_dict, associations, cost_threshold=10000):
    """
    Version améliorée du filtrage des associations avec fonctions séparées.

    Args:
        results_dict: Dictionnaire des résultats d'optimisation
        associations: Structure contenant les associations de détections
        cost_threshold: Seuil de coût pour le filtrage initial

    Returns:
        dict: Dictionnaire des résultats filtrés
    """
    print("Starting associations filtering...")
    print(f"Initial associations count: {len(results_dict)}")

    # Étape 1: Filtrer par seuil de coût
    filtered_by_cost = filter_by_cost_threshold(results_dict, cost_threshold)
    del results_dict
    # filtered_by_cost = results_dict
    # Étape 2: Résoudre les conflits
    final_results= remove_overlap_bipartite(filtered_by_cost, associations,
                                                alpha_global=0.05,
                                               alpha_outlier=0.05,
                                               min_quality=QualityLevel.GOOD,
                                               bilateral_test=False,
                                               stations=STATIONS,
                                               gap_threshold=None)
    print("associations filtering completed!")
    print(f"Final results: {len(final_results)} associations")

    return final_results

def run_filtered_associations():

    filtered_results = filter_associations_simple(results,associations)

    print(f"Processing complete:")
    print(f"  Original associations: {len(associations)}")
    print(f"  Successfully processed: {len(results)}")
    print(f"  After filtering: {len(filtered_results)}")

    return filtered_results

# Utilisation
if __name__ == "__main__":
    # import sys
    # print(sys.getrecursionlimit())
    # sys.setrecursionlimit(5000)
    final_results = run_filtered_associations()


Take a look at the results

In [ ]:
def compute_ellipsoide_error(res):
    # 1 : Calcul de la matrice de covariance
    if hasattr(res, 'jac') and res.jac is not None:
        J = res.jac
        n_obs = len(res.fun)  # Nombre d'observations
        n_params = 2  # latitude et longitude
        dof = n_obs - n_params  # Degrés de liberté
        if dof > 0:
            covariance_matrix = np.linalg.inv(J.T @ J) * (res.cost / dof)
        else:
            covariance_matrix = np.linalg.inv(J.T @ J)
    else:
        raise ValueError("Jacobien non disponible pour calculer la matrice de covariance.")

    # Calcul des valeurs propres et vecteurs propres
    eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)

    # Longueurs des axes (pour un intervalle de confiance à 95%, multiplier par 2.4477)
    confidence_level = 2.4477  # Pour 95% de confiance (2 sigma)
    semi_axes_lengths = np.sqrt(eigenvalues) * confidence_level

    # Angle d'orientation du premier axe (en degrés)
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    return semi_axes_lengths, angle

def plot_association_map_modified(results, stations_dict, title="Associations localisées"):
    """
    Carte montrant les positions des événements et des stations, avec ellipsoïdes d'erreur.
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    # Extraire les positions des événements
    event_positions = []
    costs = []
    ellipses_data = []

    for i, res in results.items():
        if hasattr(res, 'x') and len(res.x) >= 2:
            # Supposer que res.x = [t0, lat, lon] ou [lat, lon]
            if len(res.x) == 3:
                lat, lon = res.x[1], res.x[2]
            else:
                lat, lon = res.x[0], res.x[1]
            event_positions.append([lat, lon])
            cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)
            costs.append(cost)

            # Calculer l'ellipsoïde d'erreur
            try:
                semi_axes_lengths, angle = compute_ellipsoide_error(res)
                ellipses_data.append((lon, lat, semi_axes_lengths, angle))
            except Exception as e:
                print(f"Erreur pour l'événement {i}: {e}")

    if event_positions:
        event_positions = np.array(event_positions)
        costs = np.array(costs)

        # Scatter plot des événements colorés par cost
        scatter = ax.scatter(event_positions[:, 1], event_positions[:, 0],
                           c=costs, cmap='viridis_r', s=5, alpha=0.7,
                           label='Événements')
        plt.colorbar(scatter, label='Cost')

        # Ajouter les ellipsoïdes d'erreur
        for lon, lat, semi_axes_lengths, angle in ellipses_data:
            ellipse = Ellipse(
                xy=(lon, lat),
                width=2 * semi_axes_lengths[0],
                height=2 * semi_axes_lengths[1],
                angle=angle,
                edgecolor='red',
                fc='None',
                lw=1,
                alpha=0.1
            )
            ax.add_patch(ellipse)

        # Ajouter les stations si disponibles
        if stations_dict:
            station_lats = []
            station_lons = []
            for station in stations_dict:
                pos = station.get_pos()
                station_lats.append(pos[0])
                station_lons.append(pos[1])

            ax.scatter(station_lons, station_lats, c='red', marker='^',
                      s=100, label='Stations', alpha=0.8)
    ax.set_xlim(lon_min-0.2, lon_max+0.2)
    ax.set_ylim(lat_min-0.2, lat_max+0.2)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

# Exemple d'utilisation
fig_mod = plot_association_map_modified(final_results, STATIONS)
plt.show()


Export the result as a .CSV file

In [ ]:
def load_final_result(final_results):
    quality = []
    rows = []
    for i in final_results:
        res = final_results[i]
        semi_axes_lengths, angle = compute_ellipsoide_error(res)
        [a, b] = semi_axes_lengths
        aire = a*b*np.pi
        rows.append({
            "association_id": i,
            "lat": res.x[1],
            "lon": res.x[2],
            "semi_axe_length_a": a,
            "semi_axe_length_b": b,
            "angle": angle,
            "area" : aire,
            "origin_time": res.x[0],
            "cost": res.cost,
            "nb_stations": len(final_results[1].fun)+1
        })
        # quality.append(analyze_residuals(res))

    # quality = pd.DataFrame(quality)
    df = pd.DataFrame(rows)
    df["origin_time"] = pd.to_datetime(df["origin_time"], unit="s",utc=True)
    return df
df = load_final_result(final_results)
df.area.describe()
df.to_csv(f"{ASSOCIATION_OUTPUT_DIR}/catalogue_{file_name}.csv", index=False)